# V1

In [1]:
import os
import re
import time
import math
import pandas as pd
from typing import List, Dict, Optional
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import (
    NoSuchElementException, TimeoutException, StaleElementReferenceException
)
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

In [2]:
def setup_driver(headless: bool = False, window_size: str = "1200, 900") -> webdriver.Chrome:
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless=new")
        chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument(f"--window-size={window_size}")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--no-sandbox")
    # keep default user agent / do not spoof aggressively
    service = ChromeService(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    driver.set_page_load_timeout(60)
    return driver

In [3]:
def safe_find(driver, by, selector, timeout=10):
    """
    Wait up to `timeout` seconds for element to appear (polling).
    Returns element or raises TimeoutException.
    """
    t0 = time.time()
    while True:
        try:
            el = driver.find_element(by, selector)
            return el
        except Exception:
            if time.time() - t0 > timeout:
                raise TimeoutException(f"Element {selector} not found after {timeout}s")
            time.sleep(0.3)

In [4]:
def nearest_scrollable_ancestor(driver, el):
    return driver.execute_script("""
        function isScrollable(e){
          if (!e) return false;
          const cs = getComputedStyle(e);
          const oy = cs.overflowY;
          return (oy === 'auto' || oy === 'scroll') && e.scrollHeight > e.clientHeight;
        }
        let node = arguments[0];
        while (node && node !== document.body && node !== document.documentElement){
          node = node.parentElement;
          if (isScrollable(node)) return node;
        }
        return document.scrollingElement || document.documentElement || document.body;
    """, el)

In [ ]:
def scroll_to_bottom_until_stable(driver, scrollable, pause=2, max_scrolls=100000):
    last_h = -1
    for scroll_round in range(max_scrolls):
        driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight;", scrollable)
        time.sleep(pause)
        h = driver.execute_script("return arguments[0].scrollHeight;", scrollable)
        if (h == last_h) & (scroll_round>15):
            break
        last_h = h
        # print('scroll_round: ', scroll_round)


In [6]:
def fetch_reviews_for_place(place_id: str, driver: webdriver.Chrome, max_reviews: int = 100000, max_scrolls:int = 100000, verbose: bool = False) -> List[Dict]:
    """
    Open Google Maps place by place_id and extract up to max_reviews visible reviews.
    Returns a list of dicts: {place_id, author, rating, text, date}.
    """
    url = f"https://www.google.com/maps/place/?q=place_id:{place_id}"
    driver.get(url)
    time.sleep(3.0) 


    # Click on Review button
    # selectors = [
    #     "//button[contains(., 'Reviews')]",
    #     "//button[contains(@aria-label, 'reviews')]",
    #     "//div[contains(text(), 'Reviews')]",
    #     "//span[contains(text(), 'Reviews')]"
    # ]
    try:
        all_reviews_btn = safe_find(driver, 
                                    By.XPATH, 
                                    "//button[contains(., 'Reviews')]",
                                    timeout=20)
        driver.execute_script("arguments[0].click();", all_reviews_btn)
        time.sleep(1)

    except TimeoutException:
        all_reviews_btn = safe_find(driver, 
                                    By.XPATH, 
                                    "//button[contains(., 'Reviews')]",
                                    timeout=20)
        driver.execute_script("arguments[0].click();", all_reviews_btn)
        time.sleep(1.5)



    # Scrolling
    scrollable = None
    candidates = ("//div[contains(@class,'TFQHme')]"
                "| //div[contains(@class,'AyRUI')]"
                "| //div[contains(@class,'jftiEf')]",
                # "//div[contains(@class,'section-review')]",
                # " | //div[@role='article' and .//span]"
                )
    for sel in candidates:
        try:
            scrollables = driver.find_elements(By.XPATH, sel)
            break
        except Exception:
            continue
    if scrollables is None:
        scrollables = driver.find_elements(By.TAG_NAME, "body")
    scrollbox = nearest_scrollable_ancestor(driver, scrollables[-1])
    if scrollables:
        driver.execute_script("arguments[0].scrollIntoView({block:'end'});", scrollables[-1])
        scroll_to_bottom_until_stable(driver, scrollbox, pause=2, max_scrolls=max_scrolls)
    else:
        driver.execute_script("window.scrollBy(0, 400);")
    
    # time.sleep(1)



    # Scraping
    collected = []
    attempts = 0
    while len(collected) < max_reviews and attempts < 1:
        review_blocks = driver.find_elements(By.XPATH, "//div[contains(@class,'jftiEf')]")
        for rb in review_blocks:
            # print(len(collected))
            try:
                rating = None
                try:
                    star_span = rb.find_element(By.XPATH, ".//span[contains(@aria-label,'Rated') or contains(@aria-label,'star') or contains(@aria-label,'stars') or contains(@aria-label,'out of')]")
                    rating_text = star_span.get_attribute("aria-label") or star_span.text
                    m = re.search(r"(\d(\.\d)?)", rating_text)
                    if m:
                        rating = float(m.group(1))
                except Exception:
                    # try:
                    #     title_span = rb.find_element(By.XPATH, ".//span[@aria-hidden='false']")
                    #     m = re.search(r"(\d(\.\d)?)", title_span.text)
                    #     if m:
                    #         rating = float(m.group(1))
                    # except Exception:
                        continue
                        rating = None

                author = None
                try:
                    a = rb.find_element(By.XPATH, ".//div[contains(@class,'d4r55')]")
                    author = a.text.strip()
                except Exception:
                    # try:
                    #     author = rb.find_element(By.XPATH, ".//a[contains(@href,'/maps/contrib')]").text.strip()
                    # except Exception:
                        continue
                        # author = None

                date_text = None
                try:
                    d = rb.find_element(By.XPATH, ".//span[contains(@class,'rsqaWe')]")
                    date_text = d.text.strip()
                except Exception:
                    # try:
                    #     header_spans = rb.find_elements(By.XPATH, ".//div[contains(@class,'d4r55')]/following-sibling::span")
                    #     if header_spans:
                    #         date_text = header_spans[-1].text.strip()
                    # except Exception:
                        # continue
                        date_text = None

                review_text = None
                try:
                    # Google sometimes uses nested spans; try a few patterns
                    p = rb.find_element(By.XPATH, ".//span[contains(@class,'wiI7pd')]")
                    review_text = p.text.strip()
                except Exception:
                    # try:
                    #     review_text = rb.find_element(By.XPATH, ".//span[contains(@class,'review-text')]").text.strip()
                    # except Exception:
                        review_text = None


                key = (author, review_text, date_text, rating)
                if any(k for k in collected if (k.get("key") == key)):
                    continue 

                collected.append({
                    "key": key,
                    "place_id": place_id,
                    "author": author,
                    "rating": rating,
                    "date": date_text,
                    "text": review_text
                })
                if len(collected) >= max_reviews:
                    break
            except StaleElementReferenceException:
                continue
            except Exception:
                continue

        # if enough, break
        if len(collected) >= max_reviews:
            break
        time.sleep(0.8)
        attempts += 1
        # print('Attempt: ', attempts)

    out = []
    for item in collected[:max_reviews]:
        d = item.copy()
        d.pop("key", None)
        out.append(d)

    return out

In [7]:
def collect_reviews_from_excel_selenium(excel_path: str,
                                        start_row: int,
                                        # output_csv: str,
                                        place_id_col: str,
                                        place_name_col: str,
                                        max_reviews_per_place: int = 100000,
                                        headless: bool = True):
    df = pd.read_excel(excel_path).iloc[start_row:, :]
    if place_id_col is None:
        raise ValueError("Could not find a place_id column in the Excel file.")

    driver = setup_driver(headless=headless)
    
    try:
        unique_ids = df[place_id_col].dropna().astype(str).unique()
        for i, pid in enumerate(unique_ids, 1):
            start_time = time.time()  # <-- start timer

            results = []
            place_name = df.loc[df['id']==pid, place_name_col].values[0]
            safe_place_name = re.sub(r'[<>:"/\\|?*]', '_', place_name)
            place_row = df.loc[df['id']==pid, 'index'].values[0]
            filename = f"{place_row}. {safe_place_name}.xlsx"
            filepath = os.path.join("D:/PhD/Codes/Places/data/reviews", filename)

            print(f"[{i}/{len(unique_ids)}] Scraping place_id={pid} ...")
            try:
                reviews = fetch_reviews_for_place(pid, driver, max_reviews=max_reviews_per_place, verbose=True)
                if not reviews:
                    results.append({"place_id": pid, "text": "no_reviews_or_could_not_open"})
                else:
                    for r in reviews:
                        results.append(r)
            except Exception as e:
                print(f"[ERROR] place {pid}: {e}")
                results.append({"place_id": pid, "error": str(e)})
            # small delay between places to be polite
            # time.sleep(1.0)
            out_df = pd.DataFrame(results)
            out_df.to_excel(filepath)
            print(f"Iteration runtime: {time.time() - start_time:.2f} seconds") 
            print(filename)

    finally:
        driver.quit()

    print('DONE!!!')

In [8]:
if __name__ == "__main__":
    EXCEL_PATH = "D:/PhD/Codes/Places/data/places/all_places_australia_sa1.xlsx"
    MAX_REVIEWS = 1000000
    HEADLESS = True  
    START_ROW = 1394

    collect_reviews_from_excel_selenium(
        excel_path=EXCEL_PATH,
        start_row=START_ROW,
        # output_csv=OUTPUT_CSV,
        place_id_col='id',   
        place_name_col='place_name',  
        max_reviews_per_place=MAX_REVIEWS,
        headless=HEADLESS
    )

[1/246500] Scraping place_id=ChIJYbETfWiuEmsReSQ6evg8Ptg ...
Scrolling finished
Iteration runtime: 137.38 seconds
914. Searock Grill.xlsx
[2/246500] Scraping place_id=ChIJ3TaYmdY71moRbjQA9AbxvKw ...
Scrolling finished
Iteration runtime: 169.75 seconds
915. HOYTS Eastland (Ringwood).xlsx
[3/246500] Scraping place_id=ChIJLezhCVE9kWsRYMGudtc6A_w ...
Scrolling finished
Iteration runtime: 159.25 seconds
916. Curtis Falls Walking Track.xlsx
[4/246500] Scraping place_id=ChIJRaVhSQxo1moR6KiBWxXh_Qw ...
Scrolling finished
Iteration runtime: 163.91 seconds
917. Albert Park Grand Prix Circuit.xlsx
[5/246500] Scraping place_id=ChIJcd8PaXKjEmsRNpM7d1noNOE ...
Scrolling finished
Iteration runtime: 174.13 seconds
918. 500 Degrees Restaurant Parramatta.xlsx
[6/246500] Scraping place_id=ChIJGUS5tcwNnGsRJWgu7hjgTvw ...
Scrolling finished
Iteration runtime: 141.72 seconds
919. The Clog Barn Tourist Attraction & Caravan Park.xlsx
[7/246500] Scraping place_id=ChIJj-xvqSCuEmsRmNDLZn9rr0c ...
Scrolling finis

KeyboardInterrupt: 

# V2

In [12]:
import pandas as pd
import numpy as np

import os
import re
import time
import math
import pandas as pd
from typing import List, Dict, Optional
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import (
    NoSuchElementException, TimeoutException, StaleElementReferenceException
)
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [2]:
def setup_driver(headless: bool = False, window_size: str = "1200, 900") -> webdriver.Chrome:
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless=new")
        chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument(f"--window-size={window_size}")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--no-sandbox")
    # keep default user agent / do not spoof aggressively
    service = ChromeService(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    driver.set_page_load_timeout(60)
    return driver

In [3]:
def wait_click(driver, by, sel, timeout=12):
    el = WebDriverWait(driver, timeout).until(
        EC.element_to_be_clickable((by, sel))
    )
    time.sleep(0.25)
    driver.execute_script("arguments[0].click();", el)
    time.sleep(0.5)
    return el

def find_reviews_scrollbox(driver):
    # Try stable ARIA/roles first
    el = driver.execute_script("""
      // Known stable containers
      const byAria = document.querySelector('div[aria-label="Google reviews"]');
      if (byAria) return byAria;

      // Region that contains review list
      const regions = Array.from(document.querySelectorAll('div[role="region"], section[role="region"]'));
      for (const r of regions){
        if (r.querySelector('div.jftiEf')) return r;
      }

      // Fallback: parent of a review card
      const card = document.querySelector('div.jftiEf, div.TFQHme, div.AyRUI');
      if (!card) return document.scrollingElement || document.documentElement || document.body;

      function isScrollable(e){
        if (!e) return false;
        const cs = getComputedStyle(e);
        const oy = cs.overflowY;
        return (oy === 'auto' || oy === 'scroll') && e.scrollHeight > e.clientHeight;
      }
      let node = card;
      while (node && node !== document.body && node !== document.documentElement){
        node = node.parentElement;
        if (isScrollable(node)) return node;
      }
      return document.scrollingElement || document.documentElement || document.body;
    """)
    return el

In [4]:
def scroll_to_bottom_until_idle(driver, scrollable, idle_ms=1250, max_ms=100_000):
    """
    Scrolls to bottom until the container height is stable for idle_ms,
    or until max_ms is reached. Adds 0.5 sec wait between scroll steps.
    """
    driver.set_script_timeout((max_ms // 1000) + 5)
    driver.execute_async_script("""
      const sc = arguments[0];
      const idle = arguments[1];
      const maxms = arguments[2];
      const done = arguments[3];

      let lastH = -1;
      let lastGrowthAt = performance.now();
      const t0 = performance.now();

      function tick(){
        try {
          sc.scrollTop = sc.scrollHeight; // jump to bottom
        } catch(e) {}

        const h = sc.scrollHeight;
        const now = performance.now();
        if (h !== lastH){
          lastH = h;
          lastGrowthAt = now;
        }
        if ((now - lastGrowthAt) >= idle || (now - t0) >= maxms){
          return done(h);
        }
        // wait 0.5s before next frame to let lazy-load fetch more reviews
        setTimeout(() => requestAnimationFrame(tick), 500);
      }
      tick();
    """, scrollable, idle_ms, max_ms)


In [5]:
def expand_all_truncated_reviews(driver):
    # Click "More"/"Read more" buttons so you capture full text
    try:
        driver.execute_script("""
          const btns = Array.from(document.querySelectorAll('button, span'));
          for (const b of btns){
            const t = (b.innerText || '').toLowerCase();
            if (t === 'more' || t === 'read more' || t.startsWith('more'))
              b.click();
          }
        """)
    except Exception:
        pass

In [6]:
def extract_reviews_batch(driver):
    # Return array of dicts from the DOM in one go
    return driver.execute_script("""
      const cards = document.querySelectorAll("div.jftiEf");
      const out = [];
      for (const rb of cards){
        const getText = sel => {
          const el = rb.querySelector(sel);
          return el ? el.textContent.trim() : null;
        };
        const getRating = () => {
          const s = rb.querySelector("span[aria-label*='Rated'], span[aria-label*='stars'], span[aria-label*='out of']");
          if (!s) return null;
          const t = s.getAttribute("aria-label") || s.textContent || "";
          const m = t.match(/(\\d+(?:\\.\\d+)?)/);
          return m ? parseFloat(m[1]) : null;
        };
        out.push({
          author: getText(".d4r55"),
          date:   getText(".rsqaWe"),
          text:   getText(".wiI7pd"),
          rating: getRating()
        });
      }
      return out;
    """)


In [7]:
def fetch_reviews_for_place(place_id: str, 
                            driver: webdriver.Chrome, 
                            max_reviews: int = 1000000, 
                            max_scroll_ms: int = 1_000_000,
                            idle_ms: int = 1500,
                            open_timeout: int = 1000) -> List[Dict]:
    url = f"https://www.google.com/maps/place/?q=place_id:{place_id}"                      
    driver.get(url)
    # time.sleep(0.5)
    # Open the Reviews panel quickly (avoid broad XPaths, use a few robust tries)
    try:
        # common pattern: a button with inner text 'Reviews'
        wait_click(driver, By.XPATH, "//button[.//span[contains(., 'Reviews')] or contains(., 'Reviews')]", timeout=open_timeout)
    except TimeoutException:
        # fallback: aaria label variants
        wait_click(driver, By.XPATH, "//button[contains(@aria-label,'review') or contains(., 'review')]", timeout=open_timeout)

    # Find the scroll container and drive it from JS (no sleeps)
    scrollbox = find_reviews_scrollbox(driver)
    scroll_to_bottom_until_idle(driver, scrollbox, idle_ms=idle_ms, max_ms=max_scroll_ms)

    # One batched extraction from DOM
    rows = extract_reviews_batch(driver)

    # Dedupe & trim to max_reviews
    seen = set()
    out = []
    for r in rows:
        key = (r.get("author"), r.get("date"), r.get("text"), r.get("rating"))
        if key in seen: 
            continue
        seen.add(key)
        out.append({
            "place_id": place_id,
            "author": r.get("author"),
            "rating": r.get("rating"),
            "date": r.get("date"),
            "text": r.get("text")
        })
        if len(out) >= max_reviews:
            break

    return out

In [8]:
def collect_reviews_from_excel_selenium(excel_path: str,
                                        start_row: int,
                                        # output_csv: str,
                                        place_id_col: str,
                                        place_name_col: str,
                                        max_reviews_per_place: int = 100000,
                                        headless: bool = True):
    df = pd.read_excel(excel_path).iloc[start_row:, :]
    if place_id_col is None:
        raise ValueError("Could not find a place_id column in the Excel file.")

    driver = setup_driver(headless=headless)
    
    try:
        unique_ids = df[place_id_col].dropna().astype(str).unique()
        for i, pid in enumerate(unique_ids, 1):
            start_time = time.time()  # <-- start timer

            results = []
            place_name = df.loc[df['id']==pid, place_name_col].values[0]
            safe_place_name = re.sub(r'[<>:"/\\|?*]', '_', place_name)
            place_row = df.loc[df['id']==pid, 'row_num'].values[0]
            filename = f"{place_row}. {safe_place_name}.xlsx"
            filepath = os.path.join("Reviews", filename)

            print(f"[{i}/{len(unique_ids)}] Scraping place_id={pid} ...")
            try:
                reviews = fetch_reviews_for_place(pid, driver)
                if not reviews:
                    results.append({"place_id": pid, "text": "no_reviews_or_could_not_open"})
                else:
                    for r in reviews:
                        results.append(r)
            except Exception as e:
                print(f"[ERROR] place {pid}: {e}")
                results.append({"place_id": pid, "error": str(e)})
            # small delay between places to be polite
            # time.sleep(1.0)
            out_df = pd.DataFrame(results)
            out_df.to_excel(filepath)
            print(f"Iteration runtime: {time.time() - start_time:.2f} seconds") 
            print(filename)

    finally:
        driver.quit()

    print('DONE!!!')

In [ ]:
if __name__ == "__main__":
    EXCEL_PATH = "all_places_australia_sa3.xlsx" 
    MAX_REVIEWS = 1000000
    HEADLESS = False  
    START_ROW = 4748

    collect_reviews_from_excel_selenium(
        excel_path=EXCEL_PATH,
        start_row=START_ROW,
        # output_csv=OUTPUT_CSV,
        place_id_col='id',   
        place_name_col='place_name',  
        max_reviews_per_place=MAX_REVIEWS,
        headless=HEADLESS
    )

[1/2846] Scraping place_id=ChIJ60YMsb3eeaoRPr4TMxGFUTs ...
Iteration runtime: 54.80 seconds
4747. Barnbougle Dunes Golf Links.xlsx
[2/2846] Scraping place_id=ChIJ34WnfXJ4caoRRal0OtMCgcQ ...
Iteration runtime: 43.42 seconds
4748. Artifakt Gallery & Cafe.xlsx
[3/2846] Scraping place_id=ChIJuXtdf2qfPmsRjgjRY4WI1c8 ...
[ERROR] place ChIJuXtdf2qfPmsRjgjRY4WI1c8: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=140.0.7339.187)
Stacktrace:
	GetHandleVerifier [0x0x94c333+65459]
	GetHandleVerifier [0x0x94c374+65524]
	(No symbol) [0x0x76d973]
	(No symbol) [0x0x74c19d]
	(No symbol) [0x0x7e059e]
	(No symbol) [0x0x7faef9]
	(No symbol) [0x0x7d9bf6]
	(No symbol) [0x0x7ab38e]
	(No symbol) [0x0x7ac274]
	GetHandleVerifier [0x0xbceda3+2697763]
	GetHandleVerifier [0x0xbc9ec7+2677575]
	GetHandleVerifier [0x0x974194+228884]
	GetHandleVerifier [0x0x9649f8+165496]
	GetHandleVerifier [0x0x96b18d+192013]
	GetHandleVerifier [0x0x9547d8+99416]
	G

KeyboardInterrupt: 

# Test

In [ ]:
pid = 'ChIJ95ZHicUmsWoR-cSaJ_NoSsQ'
headless = False 

driver = setup_driver(headless=headless)
url = f"https://www.google.com/maps/place/?q=place_id:{pid}"
driver.get(url)
time.sleep(2.0)  # let map load

selectors = [
    "//button[contains(., 'Reviews')]",
    "//button[contains(@aria-label, 'reviews')]",
    "//div[contains(text(), 'Reviews')]",
    "//span[contains(text(), 'Reviews')]"
]

# Wait and attempt to open reviews panel:
# Google markup changes. We try multiple strategies and fail gracefully.
# try:
    # Strategy A: find "All reviews" button by aria-label (works often)
all_reviews_btn = safe_find(driver, 
                            By.XPATH, 
                            "//button[contains(., 'Reviews')]",
                            timeout=8)
driver.execute_script("arguments[0].click();", all_reviews_btn)
time.sleep(1.2)

In [21]:
collected = []
max_reviews = 100000
attempts = 0
scrollables = None
# Perform controlled scrolling until we've collected enough items or exhausted scrolling.
candidates = ("//div[contains(@class,'TFQHme')]"
            "| //div[contains(@class,'AyRUI')]"
            "| //div[contains(@class,'jftiEf')]",
            # "//div[contains(@class,'section-review')]",
            # " | //div[@role='article' and .//span]"
            )
for sel in candidates:
    try:
        scrollables = driver.find_elements(By.XPATH, sel)
        scrollable = scrollables[-1]
        break
    except Exception:
        continue
if scrollables is None:
    # fallback: try document body (less ideal)
    scrollables = driver.find_elements(By.TAG_NAME, "body")
    scrollable = scrollables[-1]

scrollbox = nearest_scrollable_ancestor(driver, scrollables[-1])

if scrollable:
    driver.execute_script("arguments[0].scrollIntoView({block:'end'});", scrollables[-1])
    scroll_to_bottom_until_stable(driver, scrollbox, pause=1.0, max_scrolls=15)

else:
    driver.execute_script("window.scrollBy(0, 400);")
time.sleep(1.2)

Scrolling finished


In [26]:
review_blocks = driver.find_elements(By.XPATH, "//div[contains(@class,'jftiEf')]")
review_blocks

[<selenium.webdriver.remote.webelement.WebElement (session="23f88fc44d13821d7789873723fdec9d", element="f.AD85A87E808E9C1208657D1F3038F2A3.d.8FD8B2E92D2C9835D8557046AF51C5CC.e.2162")>,
 <selenium.webdriver.remote.webelement.WebElement (session="23f88fc44d13821d7789873723fdec9d", element="f.AD85A87E808E9C1208657D1F3038F2A3.d.8FD8B2E92D2C9835D8557046AF51C5CC.e.2214")>,
 <selenium.webdriver.remote.webelement.WebElement (session="23f88fc44d13821d7789873723fdec9d", element="f.AD85A87E808E9C1208657D1F3038F2A3.d.8FD8B2E92D2C9835D8557046AF51C5CC.e.2271")>,
 <selenium.webdriver.remote.webelement.WebElement (session="23f88fc44d13821d7789873723fdec9d", element="f.AD85A87E808E9C1208657D1F3038F2A3.d.8FD8B2E92D2C9835D8557046AF51C5CC.e.2320")>,
 <selenium.webdriver.remote.webelement.WebElement (session="23f88fc44d13821d7789873723fdec9d", element="f.AD85A87E808E9C1208657D1F3038F2A3.d.8FD8B2E92D2C9835D8557046AF51C5CC.e.2371")>,
 <selenium.webdriver.remote.webelement.WebElement (session="23f88fc44d13821

In [27]:
len(review_blocks)

150

In [28]:
collected = []
attempts = 0
for rb in review_blocks:
            # print(len(collected))
            try:
                rating = None
                try:
                    star_span = rb.find_element(By.XPATH, ".//span[contains(@aria-label,'Rated') or contains(@aria-label,'stars') or contains(@aria-label,'out of')]")
                    rating_text = star_span.get_attribute("aria-label") or star_span.text
                    m = re.search(r"(\d(\.\d)?)", rating_text)
                    if m:
                        rating = float(m.group(1))
                except Exception:
                    # try:
                    #     title_span = rb.find_element(By.XPATH, ".//span[@aria-hidden='false']")
                    #     m = re.search(r"(\d(\.\d)?)", title_span.text)
                    #     if m:
                    #         rating = float(m.group(1))
                    # except Exception:
                        continue
                        rating = None

                author = None
                try:
                    a = rb.find_element(By.XPATH, ".//div[contains(@class,'d4r55')]")
                    author = a.text.strip()
                except Exception:
                    # try:
                    #     author = rb.find_element(By.XPATH, ".//a[contains(@href,'/maps/contrib')]").text.strip()
                    # except Exception:
                        continue
                        # author = None

                date_text = None
                try:
                    d = rb.find_element(By.XPATH, ".//span[contains(@class,'rsqaWe')]")
                    date_text = d.text.strip()
                except Exception:
                    # try:
                    #     header_spans = rb.find_elements(By.XPATH, ".//div[contains(@class,'d4r55')]/following-sibling::span")
                    #     if header_spans:
                    #         date_text = header_spans[-1].text.strip()
                    # except Exception:
                        # continue
                        date_text = None

                review_text = None
                try:
                    # Google sometimes uses nested spans; try a few patterns
                    p = rb.find_element(By.XPATH, ".//span[contains(@class,'wiI7pd')]")
                    review_text = p.text.strip()
                except Exception:
                    # try:
                    #     review_text = rb.find_element(By.XPATH, ".//span[contains(@class,'review-text')]").text.strip()
                    # except Exception:
                        review_text = None


                key = (author, review_text, date_text, rating)
                if any(k for k in collected if (k.get("key") == key)):
                    continue 

                collected.append({
                    "key": key,
                    "place_id": pid,
                    "author": author,
                    "rating": rating,
                    "date": date_text,
                    "text": review_text
                })
                if len(collected) >= max_reviews:
                    break
            except StaleElementReferenceException:
                continue
            except Exception:
                continue


In [29]:
len(collected)

128

In [16]:
rb = review_blocks[0]
rb.get_attribute('class')

''

In [ ]:
a = rb.find_element(By.XPATH, ".//div[contains(@class,'d4r55')]")
a.text

'Rodney Stuckey'

In [ ]:
while len(collected) < max_reviews and attempts < 30:
    print('len(collected): ', len(collected))
    # find review blocks currently on page
    review_blocks = driver.find_elements(By.XPATH, "//div[contains(@class,'jftiEf') or contains(@class,'section-review') or descendant::span[@aria-label]]")
    # The above XPath is deliberately broad to capture different markup versions.
    # We'll inspect each block and attempt to extract rating, text, date, author.
    for rb in review_blocks:
        print(len(collected))
        try:
            # Try different patterns to extract text and rating
            # rating: often in a span with aria-label like "5.0" or "Rated 5 out of 5"
            rating = None
            try:
                star_span = rb.find_element(By.XPATH, ".//span[contains(@aria-label,'Rated') or contains(@aria-label,'stars') or contains(@aria-label,'out of')]")
                rating_text = star_span.get_attribute("aria-label") or star_span.text
                import re
                m = re.search(r"(\d(\.\d)?)", rating_text)
                if m:
                    rating = float(m.group(1))
            except Exception:
                # try:
                #     title_span = rb.find_element(By.XPATH, ".//span[@aria-hidden='false']")
                #     m = re.search(r"(\d(\.\d)?)", title_span.text)
                #     if m:
                #         rating = float(m.group(1))
                # except Exception:
                    rating = None

            # author
            author = None
            try:
                a = rb.find_element(By.XPATH, ".//div[contains(@class,'d4r55')]")
                author = a.text.strip()
            except Exception:
                # try:
                #     author = rb.find_element(By.XPATH, ".//a[contains(@href,'/maps/contrib')]").text.strip()
                # except Exception:
                    author = None

            # date (short string like '2 months ago' or '15 Aug 2024')
            date_text = None
            try:
                d = rb.find_element(By.XPATH, ".//span[contains(@class,'rsqaWe')]")
                date_text = d.text.strip()
            except Exception:
                # try:
                #     header_spans = rb.find_elements(By.XPATH, ".//div[contains(@class,'d4r55')]/following-sibling::span")
                #     if header_spans:
                #         date_text = header_spans[-1].text.strip()
                # except Exception:
                    date_text = None

            # review text
            review_text = None
            try:
                # Google sometimes uses nested spans; try a few patterns
                p = rb.find_element(By.XPATH, ".//span[contains(@class,'wiI7pd')]")
                review_text = p.text.strip()
            except Exception:
                # try:
                #     review_text = rb.find_element(By.XPATH, ".//span[contains(@class,'review-text')]").text.strip()
                # except Exception:
                    review_text = None

            # build a stable key to avoid duplicates
            key = (author, review_text, date_text, rating)
            if any(k for k in collected if (k.get("key") == key)):
                continue  # skip duplicate

            collected.append({
                "key": key,
                "place_id": pid,
                "author": author,
                "rating": rating,
                "date": date_text,
                "text": review_text
            })
            if len(collected) >= max_reviews:
                break
        except StaleElementReferenceException:
            continue
        except Exception:
            continue

    # if enough, break
    if len(collected) >= max_reviews:
        break

    # Scroll further in the reviews container: use JS to scroll by some pixels
    try:
        driver.execute_script("arguments[0].scrollBy(0, 600);", scrollables)
    except Exception:
        # fallback to body scroll
        driver.find_element(By.TAG_NAME, "body").send_keys(Keys.PAGE_DOWN)

    time.sleep(0.8)
    attempts += 1
    print(attempts)

out = []
for item in collected[:max_reviews]:
    d = item.copy()
    d.pop("key", None)
    out.append(d)

len(collected):  0
0
1
1
2
2
2
2
2
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
4
5
6
6
6
7
8
9
9
9
10
11
12
12
12
13
14
15
15
15
16
17
18
18
18
19


In [ ]:
out = []
for item in collected[:max_reviews]:
    d = item.copy()
    d.pop("key", None)
    out.append(d)

In [ ]:
df = pd.DataFrame(out)
df.head()

,place_id,author,rating,date,text
0,ChIJzbsbvnPKcmsRyv_bOfwaEwA,Rodney Stuckey,NaN,6 months ago,Surprisingly large centre with several major d...
1,ChIJzbsbvnPKcmsRyv_bOfwaEwA,None,NaN,None,None
2,ChIJzbsbvnPKcmsRyv_bOfwaEwA,Rodney Stuckey,5.0,6 months ago,Surprisingly large centre with several major d...
3,ChIJzbsbvnPKcmsRyv_bOfwaEwA,None,5.0,6 months ago,Surprisingly large centre with several major d...
4,ChIJzbsbvnPKcmsRyv_bOfwaEwA,None,5.0,6 months ago,None


In [ ]:
df.dropna(axis=0)

,place_id,author,rating,date,text
2,ChIJzbsbvnPKcmsRyv_bOfwaEwA,Rodney Stuckey,5.0,6 months ago,Surprisingly large centre with several major d...
5,ChIJzbsbvnPKcmsRyv_bOfwaEwA,izzy,5.0,Edited 5 months ago,honestly the best shopping centre in NSW imo. ...
8,ChIJzbsbvnPKcmsRyv_bOfwaEwA,Shishir M J,5.0,Edited a year ago,I like visiting this shopping mall. It's a hug...
11,ChIJzbsbvnPKcmsRyv_bOfwaEwA,Joe Lan,5.0,7 years ago,Lovely huge shopping centre. Has everything yo...
14,ChIJzbsbvnPKcmsRyv_bOfwaEwA,Subash Kumaresan,3.0,2 months ago,Good mall but what a wasted opportunity for th...
17,ChIJzbsbvnPKcmsRyv_bOfwaEwA,Ameer Meerzaee,4.0,5 years ago,Huge shopping center that looks like an amusem...
